# **RAG con FAISS**

En este notebook implementamos el sistema de Retrieval-Augmented Generation
(RAG) para enriquecer las respuestas del LLM con contexto clínico actualizado.

La idea es sencilla: en lugar de que el modelo genere respuestas únicamente
a partir de lo que aprendió durante el fine-tuning, le proporcionamos
fragmentos relevantes de documentos clínicos reales en cada consulta.
Así las respuestas están ancladas en información verificada y el modelo
no tiene que "inventar" contexto médico.

El pipeline es:
```
Documentos .md → Chunking → Embeddings → Índice FAISS → Búsqueda por similitud → Contexto para el LLM
```

In [1]:
# Instalación de dependencias
%pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.0 MB/s eta 0:00:00


In [ ]:
#from google.colab import drive
# drive.mount('/content/drive')

import os
#os.chdir("/content/drive/MyDrive/Colab Notebooks/ECGAssistant_Sprint1")
print("Directorio:", os.getcwd())

Mounted at /content/drive
Directorio: /content/drive/.shortcut-targets-by-id/1acOp95JDTEWtgyHkn2BDLpHG-oeo8sXh/ECGAssistant_Sprint1


In [3]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

## 1. Carga de documentos clínicos

Leemos todos los archivos .md de la carpeta `rag_documents/`. Cada archivo
contiene información clínica sobre un tipo de arritmia o condición cardíaca.

In [4]:
docs_path = "rag_documents"

documentos = []
nombres    = []

for filename in sorted(os.listdir(docs_path)):
    if filename.endswith(".md"):
        filepath = os.path.join(docs_path, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            contenido = f.read()
            documentos.append(contenido)
            nombres.append(filename)

print(f"Documentos cargados: {len(documentos)}")
for n in nombres:
    print(" -", n)

Documentos cargados: 6
 - Extrasístoles_ventriculares.md
 - Fibrilación_auricular.md
 - Taquicardia supraventricular.md
 - Taquicardia_ventricular.md
 - bloqueo_av.md
 - fibrilacion_ventricular.md


## 2. Chunking


Los documentos son demasiado largos para pasarlos enteros al modelo.
Los dividimos en fragmentos (chunks) de 500 palabras con un solapamiento
de 50 palabras entre chunks consecutivos.

El solapamiento es importante para no perder contexto en los bordes de
cada fragmento: si una idea empieza al final de un chunk y termina al
principio del siguiente, el solapamiento garantiza que algún chunk la
contiene completa.

In [5]:
def split_chunks(texto, chunk_size=500, overlap=50):
    """Divide un texto en chunks de tamaño fijo con solapamiento."""
    words  = texto.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


all_chunks    = []
chunk_sources = []

for doc, nombre in zip(documentos, nombres):
    chunks = split_chunks(doc, chunk_size=500, overlap=50)
    all_chunks.extend(chunks)
    chunk_sources.extend([nombre] * len(chunks))

print(f"Total chunks generados: {len(all_chunks)}")
print(f"\nEjemplo de chunk:\n{all_chunks[0][:300]}...")

Total chunks generados: 6

Ejemplo de chunk:
# Caso clínico – Extrasístoles ventriculares (PVC) ## 1. Contexto clínico Paciente masculino de 68 años que acude a consulta por episodios intermitentes de palpitaciones y sensación de latidos irregulares. Presenta antecedentes de hipertensión arterial controlada farmacológicamente. No refiere dolor...


## 3. Generación de embeddings


Convertimos cada chunk en un vector numérico usando sentence-transformers.
Elegimos `paraphrase-multilingual-MiniLM-L12-v2` porque es gratuito,
ligero y funciona bien en español sin necesidad de GPU.

Los embeddings capturan el significado semántico de cada fragmento,
permitiendo encontrar los más relevantes para una consulta aunque no
compartan las mismas palabras exactas.

In [6]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Generando embeddings...")
embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)

print(f"Shape embeddings: {embeddings.shape}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape embeddings: (6, 384)


## 4. Índice FAISS

Con los embeddings generados construimos el índice FAISS.
Usamos `IndexFlatL2` que busca por distancia euclidiana.
Es el índice más simple y suficiente para el tamaño de nuestro corpus.

In [7]:
dimension = embeddings.shape[1]  # 384 para MiniLM

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"Índice FAISS creado con {index.ntotal} vectores")

# Guardamos el índice para no tener que regenerarlo cada vez
faiss.write_index(index, "faiss_index.bin")
print("Índice guardado en faiss_index.bin")

Índice FAISS creado con 6 vectores
Índice guardado en faiss_index.bin


## 5. Función de búsqueda

Implementamos la función de retrieval: dada una consulta en lenguaje natural,
la convierte a embedding y busca los k chunks más cercanos en el índice.

El RAG no genera respuestas, recupera los fragmentos más relevantes
del corpus clínico. Esos fragmentos se pasan como contexto al LLM,
que es el que genera la explicación final en lenguaje natural.

In [8]:
def buscar_contexto(consulta, k=3):
    """Busca los k fragmentos más relevantes para una consulta."""
    query_embedding = embedding_model.encode([consulta]).astype(np.float32)
    distancias, indices = index.search(query_embedding, k)

    resultados = []
    for i, idx in enumerate(indices[0]):
        resultados.append({
            "chunk"    : all_chunks[idx],
            "fuente"   : chunk_sources[idx],
            "distancia": distancias[0][i]
        })

    return resultados

In [9]:
# Probamos el retrieval con algunas consultas de ejemplo
consultas_prueba = [
    "fibrilación ventricular tratamiento",
    "taquicardia supraventricular diagnóstico",
    "bloqueo AV de primer grado",
]

for consulta in consultas_prueba:
    resultados = buscar_contexto(consulta, k=1)
    print(f"Consulta: {consulta}")
    print(f"Fuente recuperada: {resultados[0]['fuente']}")
    print(f"Distancia: {resultados[0]['distancia']:.4f}")
    print(f"Fragmento: {resultados[0]['chunk'][:200]}")
    print()

Consulta: fibrilación ventricular tratamiento
Fuente recuperada: fibrilacion_ventricular.md
Distancia: 13.5262
Fragmento: # Caso clínico — Fibrilación ventricular (FV) ## 1. Contexto clínico Paciente masculino de 59 años con antecedente de infarto agudo de miocardio reciente que colapsa súbitamente en su domicilio. Prese

Consulta: taquicardia supraventricular diagnóstico
Fuente recuperada: Taquicardia supraventricular.md
Distancia: 8.3174
Fragmento: # Caso clínico — Taquicardia supraventricular (TSV) ## 1. Contexto clínico Paciente masculino de 45 años que consulta por inicio súbito de palpitaciones intensas, sensación de mareo y ansiedad. Los ep

Consulta: bloqueo AV de primer grado
Fuente recuperada: bloqueo_av.md
Distancia: 18.5133
Fragmento: # Caso clínico — Bloqueo auriculoventricular (Bloqueo AV) ## 1. Contexto clínico Paciente masculino de 70 años que acude por episodios de mareo y sensación de debilidad intermitente. Refiere anteceden



## 6. Construcción del prompt RAG

Implementamos la función que integra el resultado de la CNN con el
contexto recuperado por FAISS para construir el prompt final que
recibe el LLM.

In [10]:
def generar_prompt_rag(clasificacion_cnn, k=3):
    """
    Dado el resultado de la CNN, recupera contexto clínico relevante
    y construye el prompt para el LLM.

    clasificacion_cnn: 0 (normal) o 1 (anormal)
    """
    if clasificacion_cnn == 0:
        consulta = "latido normal sinusal ECG características"
    else:
        consulta = "arritmia cardiaca diagnóstico clasificación tratamiento"

    resultados = buscar_contexto(consulta, k=k)

    contexto = "\n\n".join([r["chunk"] for r in resultados])
    fuentes  = ", ".join(set([r["fuente"] for r in resultados]))

    prompt = f"""Eres un asistente médico especializado en cardiología.
Usa solo la información del contexto para responder.

Contexto clínico:
{contexto}

### Instrucción:
El modelo de clasificación ha detectado un latido {'NORMAL' if clasificacion_cnn == 0 else 'ANORMAL'}.
Explica en español qué significa este resultado y qué implicaciones clínicas puede tener.

### Respuesta:"""

    return prompt, fuentes


# Probamos el pipeline completo con un latido anormal
prompt, fuentes = generar_prompt_rag(clasificacion_cnn=1)
print("Fuentes utilizadas:", fuentes)
print()
print(prompt[:800], "...")

Fuentes utilizadas: Taquicardia_ventricular.md, Extrasístoles_ventriculares.md, fibrilacion_ventricular.md

Eres un asistente médico especializado en cardiología. 
Usa solo la información del contexto para responder.

Contexto clínico:
# Caso clínico — Fibrilación ventricular (FV) ## 1. Contexto clínico Paciente masculino de 59 años con antecedente de infarto agudo de miocardio reciente que colapsa súbitamente en su domicilio. Presenta pérdida de conciencia y ausencia de pulso palpable. Es trasladado de urgencia al servicio de emergencias. ## 2. Evaluación mediante electrocardiograma (ECG) El ECG muestra: - Actividad eléctrica caótica - Ausencia de complejos QRS identificables - Ondas irregulares de amplitud y morfología variable - Ritmo completamente desorganizado Estos hallazgos son compatibles con fibrilación ventricular. ## Hallazgos principales en el ECG - Ausencia de ritmo organizado - No  ...


## 7. Guardado del índice y los chunks

Guardamos todo lo necesario para que el pipeline pueda cargar el índice
sin tener que regenerarlo.

In [11]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump({
        "chunks" : all_chunks,
        "sources": chunk_sources
    }, f, ensure_ascii=False)

print("Archivos guardados:")
print(" - faiss_index.bin")
print(" - chunks.json")

Archivos guardados:
 - faiss_index.bin
 - chunks.json


## Reflexión

El sistema RAG recupera correctamente los fragmentos más relevantes
según la consulta. La ventaja principal frente a incluir los documentos
enteros en el prompt es que FAISS escala bien aunque el corpus crezca,
y el modelo solo recibe el contexto realmente relevante para cada caso.

El índice y los chunks quedan guardados para que el pipeline completo
los pueda cargar directamente sin regenerarlos.